In [ ]:
# If needed, uncomment:
# !pip install -q pandas numpy matplotlib seaborn scipy

import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")

# ================================================================
# 1. LOAD DATA
# ================================================================
file_path = "Table1.csv"

try:
    df = pd.read_csv(file_path, low_memory=False)
    print(f"File loaded successfully: {file_path}")
    print(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}")
except FileNotFoundError:
    print(f"Error: file {file_path} was not found.")
    raise


# ================================================================
# 2. TRANSLATION MAPS
# ================================================================
drug_translation = {
    "Crack": "Crack",
    "Álcool": "Alcohol",
    "Alcohol": "Alcohol",
    "Cannabis": "Cannabis",
    "Cocaína": "Cocaine",
    "Cocaina": "Cocaine",
    "Tabaco": "Tobacco",
    "Outras Drogas": "Other Drugs",
    "Heroina": "Heroin",
    "Heroína": "Heroin",
    "Opioide": "Opioid",
    "Opióide": "Opioid",
}

profile_translation = {
    "Não Usuário": "Non-User",
    "Usuário Simples (1 Droga Leve)": "Single-Substance User (1 Light Drug)",
    "Poliusuário (Lícitas/Leves)": "Polysubstance User (Legal/Light Drugs)",
    "Usuário de Drogas Pesadas": "Heavy Drug User",
}

crack_status_translation = {
    "Não usa Crack": "Does Not Use Crack",
    "Usuário de Crack": "Crack User",
}

ciap_translation = {
    "K86-HIPERTENSÃO SEM COMPLICAÇÕES": "K86-HYPERTENSION WITHOUT COMPLICATIONS",
    "P01-SENSAÇÃO DE ANSIEDADE/NERVOSISMO/TENSÃO": "P01-FEELING ANXIOUS/NERVOUS/TENSE",
    "P06-PERTURBAÇÃO DO SONO": "P06-SLEEP DISTURBANCE",
    "R05-TOSSE": "R05-COUGH",
    "N01-CEFALÉIA": "N01-HEADACHE",
    "P19-ABUSO DE DROGAS": "P19-DRUG ABUSE",
    "L18-DORES MUSCULARES": "L18-MUSCLE PAIN",
    "P76-DEPRESSÃO": "P76-DEPRESSION",
    "R02-FALTA DE AR/DISPNEIA": "R02-SHORTNESS OF BREATH/DYSPNEA",
    "A04-FRAQUEZA/CANSAÇO GERAL": "A04-GENERAL WEAKNESS/FATIGUE",
    "A03-FEBRE": "A03-FEVER",
    "D01-DOR ABDOMINAL/CÓLICA": "D01-ABDOMINAL PAIN/CRAMPS",
    "L03-SINTOMAS/QUEIXAS LOMBARES": "L03-LOW BACK SYMPTOMS/COMPLAINTS",
    "R74-INFECÇÃO AGUDA DO TRATO RESPIRATÓRIO SUPERIOR": "R74-ACUTE UPPER RESPIRATORY TRACT INFECTION",
    "T90-DIABETES NÃO INSULINO-DEPENDENTE": "T90-NON-INSULIN-DEPENDENT DIABETES",
    "P99-OUTRAS PERTURBAÇÕES PSICOLÓGICAS": "P99-OTHER PSYCHOLOGICAL DISORDERS",
    "P98-OUTRAS PSICOSES NE": "P98-OTHER PSYCHOSES, NOT ELSEWHERE CLASSIFIED",
    "P74-DISTÚRBIO ANSIOSO/ESTADO DE ANSIEDADE": "P74-ANXIETY DISORDER / ANXIETY STATE",
    "P72-ESQUIZOFRENIA": "P72-SCHIZOPHRENIA",
}

def translate_drug_name(value: str) -> str:
    value = str(value).strip()
    return drug_translation.get(value, value)

def translate_profile(value: str) -> str:
    value = str(value).strip()
    return profile_translation.get(value, value)

def translate_crack_status(value: str) -> str:
    value = str(value).strip()
    return crack_status_translation.get(value, value)

def translate_ciap(value: str) -> str:
    value = str(value).strip()
    return ciap_translation.get(value, value)


# ================================================================
# 3. FIGURE 1 — POLYSUBSTANCE USE VS PSYCHIATRY REFERRAL
# ================================================================
def figure_1_polysubstance_vs_psychiatry(df: pd.DataFrame) -> None:
    target_drugs = ["Crack", "Álcool", "Cannabis", "Cocaína", "Tabaco", "Outras Drogas"]
    drug_cols = [c for c in df.columns if "choice=Sim" in c and any(d in c for d in target_drugs)]

    psychiatry_cols = [c for c in df.columns if "Para qual especialidade" in c and "choice=Psiquiatria" in c]

    if not psychiatry_cols:
        print("ERROR: Psychiatry referral column was not found.")
        return

    col_psychiatry = psychiatry_cols[0]

    df_local = df.sort_values(["Record ID", "Repeat Instance"], na_position="first").copy()
    cols_to_fill = drug_cols + [col_psychiatry]
    df_local[cols_to_fill] = df_local.groupby("Record ID")[cols_to_fill].ffill()

    df_visits = df_local[df_local["Repeat Instrument"].notna()].copy()

    df_visits["Number_of_Drugs"] = (df_visits[drug_cols] == "Checked").sum(axis=1)
    df_visits["Referred_to_Psychiatry"] = (df_visits[col_psychiatry] == "Checked").astype(int)

    export_table = df_visits.groupby("Number_of_Drugs").agg(
        Total_Visits=("Referred_to_Psychiatry", "count"),
        Psychiatry_Referrals=("Referred_to_Psychiatry", "sum"),
    ).reset_index()

    export_table["Referral_Rate (%)"] = (
        export_table["Psychiatry_Referrals"] / export_table["Total_Visits"] * 100
    ).round(2)

    export_table.columns = [
        "Polysubstance Index (No. of Drugs)",
        "Total Visits (N)",
        "Psychiatry Referrals (N)",
        "Referral Rate (%)",
    ]
    export_file_1 = "polysubstance_psychiatry_base_en.csv"
    export_table.to_csv(export_file_1, index=False, encoding="utf-8-sig")

    corr_coef, p_value = stats.pointbiserialr(
        df_visits["Referred_to_Psychiatry"],
        df_visits["Number_of_Drugs"]
    )

    stats_table = pd.DataFrame({
        "Metric": [
            "Point-Biserial Correlation (r)",
            "P-Value (Significance)",
            "Total N Analyzed",
        ],
        "Value": [round(corr_coef, 4), p_value, len(df_visits)],
    })
    export_file_2 = "polysubstance_statistics_en.csv"
    stats_table.to_csv(export_file_2, index=False, encoding="utf-8-sig")

    print("\n" + "=" * 80)
    print("DATA TABLES EXPORTED SUCCESSFULLY:")
    print(f"- '{export_file_1}' (chart data: N and rates)")
    print(f"- '{export_file_2}' (correlation test results)")
    print("=" * 80)

    referral_rate = export_table.set_index("Polysubstance Index (No. of Drugs)")["Referral Rate (%)"]
    case_volume = export_table.set_index("Polysubstance Index (No. of Drugs)")["Total Visits (N)"]

    fig, ax1 = plt.subplots(figsize=(10, 6))
    sns.set_theme(style="whitegrid")

    sns.barplot(x=referral_rate.index, y=referral_rate.values, color="#3498db", ax=ax1)
    ax1.set_title(
        "Impact of Polysubstance Use on Psychiatry Referral",
        fontsize=14,
        pad=15,
        fontweight="bold",
    )
    ax1.set_xlabel("Number of Substances Used (Polysubstance Index)", fontsize=12)
    ax1.set_ylabel("Psychiatry Referral Rate (%)", fontsize=12, color="#2980b9")
    ax1.tick_params(axis="y", labelcolor="#2980b9")

    for patch in ax1.patches:
        if patch.get_height() > 0:
            ax1.annotate(
                f"{patch.get_height():.1f}%",
                (patch.get_x() + patch.get_width() / 2.0, patch.get_height()),
                ha="center",
                va="center",
                fontsize=11,
                fontweight="bold",
                color="black",
                xytext=(0, 8),
                textcoords="offset points",
            )

    ax2 = ax1.twinx()
    sns.lineplot(
        x=case_volume.index,
        y=case_volume.values,
        color="#e74c3c",
        marker="o",
        linewidth=2.5,
        ax=ax2,
    )
    ax2.set_ylabel("Total Visit Volume (N)", fontsize=12, color="#c0392b")
    ax2.tick_params(axis="y", labelcolor="#c0392b")
    ax2.grid(False)

    plt.tight_layout()
    plt.show()

    print("\n" + "=" * 80)
    print("STATISTICAL SUMMARY: POLYSUBSTANCE USERS VS PSYCHIATRY")
    print("=" * 80)
    print(export_table.to_string(index=False))
    print(f"\nTotal Visits Analyzed: {len(df_visits)}\n")
    print("POINT-BISERIAL CORRELATION TEST METRICS:")
    print(f" -> Correlation Coefficient (r): {corr_coef:.4f}")
    print(f" -> P-value (Significance):      {p_value:.4e}")
    print("=" * 80)


# ================================================================
# 4. FIGURE 2 — TRIPLE PANEL: DUAL PATHOLOGY BY UNIQUE PATIENT
# ================================================================
def figure_2_dual_pathology_panel(df: pd.DataFrame) -> None:
    text_cols = [c for c in df.columns if df[c].dtype == object]

    df_local = df.copy()

    df_local["Use_Crack"] = df_local[text_cols].apply(
        lambda x: x.astype(str).str.contains(r"\bcrack\b", case=False, na=False)
    ).any(axis=1).astype(int)
    df_local["Use_Cocaine"] = df_local[text_cols].apply(
        lambda x: x.astype(str).str.contains(r"\bcoca[íi]na\b", case=False, na=False)
    ).any(axis=1).astype(int)
    df_local["Use_Cannabis"] = df_local[text_cols].apply(
        lambda x: x.astype(str).str.contains(r"\b(maconha|cannabis)\b", case=False, na=False)
    ).any(axis=1).astype(int)
    df_local["Use_Heroin"] = df_local[text_cols].apply(
        lambda x: x.astype(str).str.contains(r"\bhero[íi]na\b", case=False, na=False)
    ).any(axis=1).astype(int)
    df_local["Use_Opioid"] = df_local[text_cols].apply(
        lambda x: x.astype(str).str.contains(r"\bopi[óo]ide\b", case=False, na=False)
    ).any(axis=1).astype(int)

    alcohol_cols = [c for c in df_local.columns if ("álcool" in c.lower() or "alcool" in c.lower()) and "choice=" in c.lower()]
    tobacco_cols = [c for c in df_local.columns if ("tabaco" in c.lower() or "fumo" in c.lower() or "cigarro" in c.lower()) and "choice=" in c.lower()]

    df_local["Use_Alcohol"] = (
        df_local[alcohol_cols]
        .apply(lambda x: x.astype(str).str.contains("Checked|1|sim", case=False, na=False))
        .any(axis=1).astype(int)
        if alcohol_cols else 0
    )
    df_local["Use_Tobacco"] = (
        df_local[tobacco_cols]
        .apply(lambda x: x.astype(str).str.contains("Checked|1|sim", case=False, na=False))
        .any(axis=1).astype(int)
        if tobacco_cols else 0
    )

    psych_cols = [
        c for c in df_local.columns
        if ("encaminhamento" in c.lower() or "especialidade" in c.lower())
        and ("psiquiatria" in c.lower() or "saúde mental" in c.lower() or "psicologia" in c.lower())
    ]

    if psych_cols:
        df_local["Referred_Psych"] = (
            df_local[psych_cols]
            .apply(lambda x: x.astype(str).str.contains("Checked|1|sim", case=False, na=False))
            .any(axis=1).astype(int) * 100
        )
    else:
        df_local["Referred_Psych"] = 0
        print("WARNING: Psychiatry referral column was not found.")

    analysis_cols = [
        "Use_Crack", "Use_Cocaine", "Use_Cannabis",
        "Use_Heroin", "Use_Opioid", "Use_Alcohol",
        "Use_Tobacco", "Referred_Psych"
    ]

    df_user = df_local.groupby("Record ID")[analysis_cols].max().reset_index()

    all_drugs = [
        "Use_Crack", "Use_Cocaine", "Use_Cannabis",
        "Use_Heroin", "Use_Opioid", "Use_Alcohol", "Use_Tobacco"
    ]
    heavy_drugs = ["Use_Crack", "Use_Cocaine", "Use_Heroin", "Use_Opioid"]

    df_user["Total_Drugs_Used"] = df_user[all_drugs].sum(axis=1)
    df_user["Used_Heavy_Drug"] = df_user[heavy_drugs].sum(axis=1) > 0

    def classify_profile(row):
        if row["Total_Drugs_Used"] == 0:
            return "Non-User"
        if row["Used_Heavy_Drug"]:
            return "Heavy Drug User"
        if row["Total_Drugs_Used"] > 1:
            return "Polysubstance User (Legal/Light Drugs)"
        return "Single-Substance User (1 Light Drug)"

    df_user["Clinical_Profile"] = df_user.apply(classify_profile, axis=1)
    df_user["Crack_Status"] = df_user["Use_Crack"].apply(
        lambda x: "Crack User" if x == 1 else "Does Not Use Crack"
    )

    ranking = df_user[all_drugs].sum().sort_values(ascending=False)
    ranking = ranking[ranking > 0]

    ranking_names = []
    for idx in ranking.index:
        raw = idx.replace("Use_", "")
        ranking_names.append(translate_drug_name(raw))
    ranking.index = ranking_names

    ranking_df = ranking.reset_index()
    ranking_df.columns = ["Substance", "Unique Patients (N)"]
    ranking_file = "substance_ranking_en.csv"
    ranking_df.to_csv(ranking_file, index=False, encoding="utf-8-sig")

    profile_table = df_user.groupby("Clinical_Profile")["Referred_Psych"].agg(["mean", "std", "count"]).reset_index()
    profile_table.columns = [
        "Clinical Profile",
        "Psychiatry Referral Rate (%)",
        "Standard Deviation",
        "N (Patients)",
    ]
    profile_order = [
        "Non-User",
        "Single-Substance User (1 Light Drug)",
        "Polysubstance User (Legal/Light Drugs)",
        "Heavy Drug User",
    ]
    profile_table["Clinical Profile"] = pd.Categorical(
        profile_table["Clinical Profile"],
        categories=profile_order,
        ordered=True,
    )
    profile_table = profile_table.sort_values("Clinical Profile")
    profile_file = "drug_profile_psychiatry_en.csv"
    profile_table.to_csv(profile_file, index=False, encoding="utf-8-sig")

    crack_table = df_user.groupby("Crack_Status")["Referred_Psych"].agg(["mean", "std", "count"]).reset_index()
    crack_table.columns = [
        "Crack Use Status",
        "Psychiatry Referral Rate (%)",
        "Standard Deviation",
        "N (Patients)",
    ]
    crack_order = ["Does Not Use Crack", "Crack User"]
    crack_table["Crack Use Status"] = pd.Categorical(
        crack_table["Crack Use Status"],
        categories=crack_order,
        ordered=True,
    )
    crack_table = crack_table.sort_values("Crack Use Status")
    crack_file = "crack_factor_psychiatry_en.csv"
    crack_table.to_csv(crack_file, index=False, encoding="utf-8-sig")

    print("\n" + "=" * 80)
    print(f"DATA TABLES EXPORTED SUCCESSFULLY (N = {len(df_user)} UNIQUE PATIENTS):")
    print(f"- '{ranking_file}'")
    print(f"- '{profile_file}'")
    print(f"- '{crack_file}'")
    print("=" * 80)

    sns.set_theme(style="whitegrid")
    fig = plt.figure(figsize=(18, 12))

    ax1 = plt.subplot2grid((2, 2), (0, 0), colspan=2)
    ax2 = plt.subplot2grid((2, 2), (1, 0))
    ax3 = plt.subplot2grid((2, 2), (1, 1))

    ranking_colors = [
        "#c0392b" if d in ["Crack", "Cocaine", "Heroin", "Opioid"] else "#7f8c8d"
        for d in ranking.index
    ]

    sns.barplot(x=ranking.values, y=ranking.index, palette=ranking_colors, ax=ax1)
    ax1.set_title(
        "Ranking of Most Used Substances (UNIQUE Patients)",
        fontsize=15,
        fontweight="bold",
        pad=15,
    )
    ax1.set_xlabel("Number of Unique Patients", fontsize=12, fontweight="bold")
    ax1.set_ylabel("")

    if psych_cols:
        sns.barplot(
            data=df_user,
            x="Clinical_Profile",
            y="Referred_Psych",
            order=profile_order,
            palette=["#2ecc71", "#f1c40f", "#e67e22", "#c0392b"],
            capsize=0.05,
            err_kws={"linewidth": 1.5},
            ax=ax2,
        )
        ax2.set_title(
            "Cascade Effect: Polysubstance Use and Heavy Drugs on Psychiatry",
            fontsize=14,
            fontweight="bold",
            pad=15,
        )
        ax2.set_ylabel("% of Patients Referred to Psychiatry", fontsize=12, fontweight="bold")
        ax2.set_xlabel("")
        ax2.tick_params(axis="x", rotation=15)

        sns.barplot(
            data=df_user,
            x="Crack_Status",
            y="Referred_Psych",
            order=crack_order,
            palette=["#95a5a6", "#8e44ad"],
            capsize=0.1,
            err_kws={"linewidth": 1.5},
            ax=ax3,
        )

        for patch in ax3.patches:
            ax3.annotate(
                f"{patch.get_height():.1f}%",
                (patch.get_x() + patch.get_width() / 2.0, patch.get_height()),
                ha="center",
                va="center",
                xytext=(0, 10),
                textcoords="offset points",
                fontweight="bold",
            )

        ax3.set_title(
            "The Crack Factor (Dual Pathology)",
            fontsize=14,
            fontweight="bold",
            pad=15,
        )
        ax3.set_ylabel("% of Patients Referred to Psychiatry", fontsize=12, fontweight="bold")
        ax3.set_xlabel("")

    sns.despine(bottom=True, left=True)
    plt.tight_layout(pad=3.0)
    plt.show()


# ================================================================
# 5. FIGURE 3 — PSYCHIATRY QUEUE X-RAY (CIAP-2)
# ================================================================
def figure_3_psychiatry_queue_xray(df: pd.DataFrame) -> None:
    specialty_cols = [c for c in df.columns if ("encaminhamento" in c.lower() or "especialidade" in c.lower()) and "choice=" in c.lower()]
    ciap_cols = [c for c in df.columns if "ciap" in c.lower()]

    psych_col = next(
        (c for c in specialty_cols if "psiquiatria" in c.lower() or "saúde mental" in c.lower() or "psicologia" in c.lower()),
        None
    )

    if not psych_col:
        print("ERROR: Psychiatry/Mental Health column was not found.")
        return
    if not ciap_cols:
        print("ERROR: Consultation Reason (CIAP-2) column was not found.")
        return

    target_ciap_col = ciap_cols[0]

    is_psych = df[psych_col].astype(str).str.lower().str.strip().isin(
        ["checked", "1", "1.0", "sim", "verdadeiro"]
    )
    df_psych = df[is_psych].copy()

    total_psych_referrals = len(df_psych)

    if total_psych_referrals == 0:
        print("No Psychiatry referrals were found using these criteria.")
        return

    psych_reasons = df_psych[target_ciap_col].dropna().value_counts().reset_index()
    psych_reasons.columns = ["Consultation Reason (CIAP-2)", "Absolute Frequency (N)"]
    psych_reasons["Proportion (%)"] = (
        psych_reasons["Absolute Frequency (N)"] / total_psych_referrals * 100
    ).round(2)

    psych_reasons["Consultation Reason (CIAP-2)"] = psych_reasons["Consultation Reason (CIAP-2)"].apply(translate_ciap)

    export_file = "psychiatry_reasons_base_en.csv"
    psych_reasons.to_csv(export_file, index=False, encoding="utf-8-sig")

    print("\n" + "=" * 80)
    print("DATA TABLE EXPORTED SUCCESSFULLY:")
    print(f"- '{export_file}' (complete Psychiatry queue with N and proportion)")
    print("=" * 80)

    top_reasons = psych_reasons.head(7).copy()
    top_reasons["Short_Name"] = top_reasons["Consultation Reason (CIAP-2)"].apply(
        lambda m: str(m)[:65] + ("..." if len(str(m)) > 65 else "")
    )

    chart_values = top_reasons["Absolute Frequency (N)"].values
    chart_labels = top_reasons["Short_Name"].values

    sns.set_theme(style="whitegrid")
    plt.figure(figsize=(14, 7))

    palette_reasons = ["#e74c3c"] + ["#bdc3c7"] * (len(top_reasons) - 1)
    ax = sns.barplot(x=chart_values, y=chart_labels, palette=palette_reasons)

    plt.title(
        f"Psychiatry X-Ray: Why are patients being referred?\n(Analysis base: {total_psych_referrals} referrals issued)",
        fontsize=16,
        fontweight="bold",
        pad=20,
    )
    plt.xlabel("Absolute Case Volume", fontsize=13, fontweight="bold")
    plt.ylabel("Consultation Reason (CIAP-2 Code)", fontsize=13, fontweight="bold")

    for patch in ax.patches:
        qty = int(patch.get_width())
        pct = (qty / total_psych_referrals) * 100
        ax.annotate(
            f"  {qty} cases ({pct:.1f}%)",
            (qty, patch.get_y() + patch.get_height() / 2.0),
            ha="left",
            va="center",
            xytext=(5, 0),
            textcoords="offset points",
            fontweight="bold",
            fontsize=11,
            color="#2c3e50",
        )

    sns.despine(left=True, bottom=True)
    plt.xlim(0, max(chart_values) * 1.20)
    plt.tight_layout()
    plt.show()

    print("\n" + "=" * 90)
    print(f"PSYCHIATRY QUEUE DIAGNOSIS (N = {total_psych_referrals})")
    print("=" * 90)

    champion_reason = psych_reasons.iloc[0]["Consultation Reason (CIAP-2)"]
    champion_pct = psych_reasons.iloc[0]["Proportion (%)"]

    print(f"The main driver of the Psychiatry queue in TeleSAP is:\n-> '{champion_reason}'\n")
    print(f"This single reason accounts for {champion_pct:.1f}% of ALL referrals to the specialty.")

    if "ANXIOUS" in champion_reason.upper() or "DEPRESSION" in champion_reason.upper():
        print("\n[RESOLUTION CAPACITY ALERT]")
        print("Since the leading reason involves a Common Mental Disorder (Anxiety / Mild to Moderate Depression),")
        print("there is strong potential for resolution within Primary Care itself, as long as physicians receive")
        print("training and clear protocols for SSRI prescription and initial clinical management.")
    print("=" * 90 + "\n")


# ================================================================
# 6. RUN EVERYTHING
# ================================================================
def main():
    figure_1_polysubstance_vs_psychiatry(df)
    figure_2_dual_pathology_panel(df)
    figure_3_psychiatry_queue_xray(df)


main()